In [1]:
!pip install -q -U "transformers>=4.45.0,<5.0.0" huggingface_hub datasets peft trl bitsandbytes accelerate wandb

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 97.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 34.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 515.2/515.2 kB 32.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 540.5/540.5 kB 30.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 31.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.3/25.3 MB 66.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 38.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.31.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.21.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.


In [2]:
import wandb
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
login(token=user_secrets.get_secret("HF_TOKEN"))
wandb.login(key=user_secrets.get_secret("WANDB_API_KEY"))

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: shreyashgaur01 (Nyaya-LLM) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [3]:
import wandb
wandb.init(
    entity="Nyaya-LLM",
    project="nyaya-llm-phase1",
    name="phase2-lora-gemma-3-4b", 
)

import torch
import os
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, get_peft_model
from trl import SFTTrainer, SFTConfig

MODEL_NAME = "google/gemma-3-4b-it"
DATA_PATH = "/kaggle/input/datasets/shreyashgaur222/nyaya-llm-dataset/nyaya_phase2_mixed_train.jsonl"
VAL_PATH = "/kaggle/input/datasets/shreyashgaur222/nyaya-llm-dataset/all_acts_val.jsonl"
OUTPUT_DIR = "/kaggle/working/results/lora_phase2_gemma3_4b"
MAX_SEQ_LENGTH = 512

def format_prompt(example):
    return f"### Instruction:\n{example['instruction']}\n\n### Response:\n{example['output']}"

class GemmaSFTTrainer(SFTTrainer):
    """Injects token_type_ids to prevent Gemma-3 multimodal crashes."""
    def training_step(self, model, inputs, num_items_in_batch=None):
        if "token_type_ids" not in inputs:
            inputs["token_type_ids"] = torch.zeros_like(inputs["input_ids"])
        return super().training_step(model, inputs, num_items_in_batch)
    def prediction_step(self, model, inputs, prediction_loss_only, ignore_keys=None):
        if "token_type_ids" not in inputs:
            inputs["token_type_ids"] = torch.zeros_like(inputs["input_ids"])
        return super().prediction_step(model, inputs, prediction_loss_only, ignore_keys)

def main():
    print("Loading dataset...")
    dataset = load_dataset("json", data_files={"train": DATA_PATH, "validation": VAL_PATH})

    print("Loading tokenizer...")
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=False)
    tokenizer.pad_token = tokenizer.eos_token

    print("Loading base model in fp32...")
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        torch_dtype="auto",
        device_map="auto",
        attn_implementation="eager",
        trust_remote_code=False,
    )

    model.gradient_checkpointing_enable()
    model.enable_input_require_grads()
    model.config.use_cache = False

    print("Applying LoRA adapters...")
    peft_config = LoraConfig(
        r=16,
        lora_alpha=32,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
        lora_dropout=0.05,
        bias="none",
        task_type="CAUSAL_LM",
    )
    model = get_peft_model(model, peft_config)
    model.print_trainable_parameters()

    print("Setting up Trainer...")
    training_args = SFTConfig(
        output_dir=OUTPUT_DIR,
        per_device_train_batch_size=1,
        per_device_eval_batch_size=1,
        gradient_accumulation_steps=8,
        eval_strategy="steps",
        eval_steps=100,
        logging_steps=25,
        num_train_epochs=2,
        learning_rate=2e-4,
        fp16=False,
        bf16=False,
        max_grad_norm=1.0,
        warmup_ratio=0.03,
        optim="adamw_torch",
        save_strategy="steps",
        save_steps=825,
        save_total_limit=5,
        max_length=MAX_SEQ_LENGTH,
        report_to="wandb",
        dataloader_pin_memory=False,
    )

    trainer = GemmaSFTTrainer(
        model=model,
        processing_class=tokenizer,
        train_dataset=dataset["train"],
        eval_dataset=dataset["validation"],
        formatting_func=format_prompt,
        args=training_args,
    )

    example = dataset["train"][0]
    text = format_prompt(example)
    batch = tokenizer(
        [text],
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=MAX_SEQ_LENGTH,
    ).to("cuda")
    with torch.no_grad():
        out = model(**batch, labels=batch["input_ids"])
    print("Manual loss:", out.loss.item())

    print("Starting training...")
    trainer.train()

    print("Saving model adapter...")
    trainer.model.save_pretrained(OUTPUT_DIR)
    tokenizer.save_pretrained(OUTPUT_DIR)
    print("Gemma-3-4B LoRA Training Complete.")

if __name__ == "__main__":
    main()

wandb: Tracking run with wandb version 0.25.0
wandb: Run data is saved locally in /kaggle/working/wandb/run-20260224_195410-vciyhlqk
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run phase2-lora-gemma-3-4b
wandb: ⭐️ View project at https://wandb.ai/Nyaya-LLM/nyaya-llm-phase1
wandb: 🚀 View run at https://wandb.ai/Nyaya-LLM/nyaya-llm-phase1/runs/vciyhlqk
2026-02-24 19:54:30.254471: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1771962870.454116      24 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1771962870.509100      24 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1771962870.976767      24 computation_placer.cc:177] 

Loading dataset...


Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Loading tokenizer...


...

Truncating eval dataset:   0%|          | 0/969 [00:00<?, ? examples/s]

The model is already on multiple devices. Skipping the move to device specified in `args`.
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 1, 'bos_token_id': 2, 'pad_token_id': 1}.


Manual loss: 2.5396389961242676
Starting training...


Step,Training Loss,Validation Loss
100,1.315100,1.263691
200,1.231700,1.188284
300,1.207600,1.148580
400,1.222300,1.128778
500,1.167600,1.094087
600,1.116500,1.080904
700,1.108500,1.058416
800,1.101200,1.037152
900,1.079500,1.019963
1000,1.040200,1.011911


Saving model adapter...
Gemma-3-4B LoRA Training Complete.
